# 04 Operate traces and audit
## Learning objective
Inspect correlation, OTel spans, completeness and tamper detection without confusing application evidence with native product signals.
## Technical level
L300-L500. Expected duration: 15 minutes.
## Prerequisites and setup
Notebook 00; Python 3.12 `.venv`. Bash/PowerShell: `uv sync --frozen --all-extras --dev`. Read the [notebook guide](README.md) for lab order, consent rules and the production/WAF disclaimer.
## Cost and permissions
These cells are offline and require no Azure permissions. Optional cloud telemetry is separately authorized, billable, and not executed here.


In [ ]:
from pathlib import Path
from uuid import uuid4

from foundry_security_accelerator.audit import audit_failure_line, verify_audit
from foundry_security_accelerator.config import Settings
from foundry_security_accelerator.contracts import Outcome, UserRequest
from foundry_security_accelerator.evaluation import trace_completeness
from foundry_security_accelerator.observability import read_stage_names
from foundry_security_accelerator.workflow import REQUIRED_STAGES, run

ROOT = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
directory = ROOT / ".fsa" / "notebooks" / str(uuid4())
result = run(
    UserRequest(text="Explain equipment policy.", subject="synthetic-analyst"),
    directory,
    settings=Settings.local(),
    demo_approval=True,
)
assert result["telemetry_export"] == "LOCAL_ONLY"
assert result["correlation_id"] and len(result["traceparent"]) == 55
assert (
    trace_completeness(REQUIRED_STAGES, read_stage_names(Path(result["traces"]))).outcome
    == Outcome.PASS
)
assert verify_audit(Path(result["audit"]), result["audit_head"])
original = Path(result["audit"]).read_text()
tampered = directory / "tampered-copy.jsonl"
tampered.write_text(original.replace("input-security", "input-securitx", 1))
assert audit_failure_line(tampered) == 2
assert verify_audit(Path(result["audit"]), result["audit_head"])
print(
    {"export": result["telemetry_export"], "complete": True, "tampered_copy_first_broken_line": 2}
)

## Optional read-only cloud observation (manual)
See [trace architecture](../docs/observability/trace-architecture.md) for private Application Insights settings and export consent. Never display connection strings. `FSA_EXPORT_MONITOR` enables export only in Azure mode; selecting it is not ingestion evidence. With approved workspace query permissions and your own configured environment, run `uv run fsa observe --correlation-id VALUE` in a private terminal using an actual run value. This notebook does not query Azure.
## Expected results and validation
LOCAL_ONLY, complete stages for the explicitly approved DRY_RUN case, verified original audit and tampered-copy failure at line 2. A normal AWAITING_APPROVAL run intentionally lacks approval/writer stages, so a generic all-stage completeness check need not pass.
## Security explanation
A hash chain needs an independently retained head/anchor against wholesale rewriting. Local traces, export attempts, observed destination rows, Agent 365 activity, Defender alerts and Purview events are six distinct evidence scopes. Never promote one into another. Report centrally; enforce natively in each cloud.
## Troubleshooting
No rows or partial cloud query: insufficient evidence, not proof nothing happened. Export failure: preserve local evidence and investigate; do not claim delivery succeeded.
## Cleanup
Never modify original audit files. Keep only the named tampered copy for teaching, and clear outputs. [Cleanup](../docs/participant-lab/cleanup.md).
## Source modules
[observability](../src/foundry_security_accelerator/observability.py), [audit](../src/foundry_security_accelerator/audit.py), [workflow](../src/foundry_security_accelerator/workflow.py).
